In [24]:
from sm import config
import pandas as pd

# load data
df = pd.read_csv(
    config.data_dir / "EAGx_Amsterdam_11_12_25.csv",
    skiprows=5,
)
# lower case all column names
df.columns = df.columns.str.lower()
df.drop(columns=["first name", "last name", "swapcard", "linkedin"], inplace=True)
df.rename(
    columns={
        "how others can help me": "help_me",
        "how i can help others": "help_others",
        "job title": "job",
        "areas of expertise": "expertise",
        "areas of interest": "interests",
    },
    inplace=True,
)
df.head()


,company,job,career stage,biography,expertise,interests,help_me,help_others
0,initiative for safe migration and social justice,Data Analyst,"Not employed, but looking; Pursuing an undergr...","I am Ghaniyyah Abdulkareem, a passionate advoc...",Data science/Data visualization; Wild animal w...,Climate change mitigation; Entrepreneurship; E...,"I’m hoping to gain deeper insights, new connec...",I bring a unique blend of grassroots experienc...
1,Oxford University,Cultivated Meat & Animal Welfare Researcher,"Not employed, but looking",Exploring founding in the global health or alt...,Academic research; Alternative proteins; Farme...,Academic research; Alternative proteins; Biose...,Feedback on Global Health Intervention ideas. ...,Brainstorm ideas. \nOffer biochemistry experti...
2,NPL,Climate Data Scientist,"Working (0–5 years experience); Employed, full...",I currently plan to try and personally improve...,Academic research; Climate change mitigation; ...,Academic research; Civilisational recovery/res...,"Connect with people, and gain a deeper insight...",I can (potentially!!) provide some discussion ...
3,Lagos State Ministry of Education,Education Officer,Working (6–15 years of experience),As a dedicated and passionate Science tutor an...,Academic research; Climate change mitigation; ...,Academic research; AI strategy & policy; Clima...,NaN,NaN
4,NaN,NaN,Pursuing an undergraduate degree,NaN,Academic research; AI safety technical researc...,Academic research; AI safety technical researc...,NaN,NaN


# Traditional analysis

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")


def extract_locations(row: pd.Series, col_subset: list | list[str]):
    """Extract locations from a row of a dataframe from the relevant provided column(s).

    Args:
        row: pd.Series
        col_subset (str | list[str]): The column(s) to extract locations from. If a string is provided, it will be converted to a list.
    Returns:
        list[str]: A list of locations extracted from the provided column(s).
    """

    def extract_location_from_text(text):
        doc = nlp(text)
        return [ent.text for ent in doc.ents if ent.label_ == "GPE"]

    locations = []
    if isinstance(col_subset, str):
        col_subset = [col_subset]
    for col in col_subset:
        text = row[col] if pd.notnull(row[col]) else ""
        locations.extend(extract_location_from_text(text))
    return locations


df["locations"] = df.apply(extract_locations, col_subset="biography", axis=1)

# LLM analysis

# Deprecated working

In [20]:
### DEPRECATED

# import uuid

# # anonymise (unique ID to replace First name/Last name columns)
# df["UID"] = df.apply(lambda row: f"{row['first name']}_{row['last name']}", axis=1)
# # # replace UID with a random string
# df["UID"] = df["UID"].apply(lambda x: str(uuid.uuid4()))
# df.head()
